# NLP Lab 2 Assignment: Sentence Generation using N-Grams

In this notebook we build **Bigram** and **Trigram** language models from a corpus, and generate
whole sentences word by word.

For each N-gram size we generate sentences **two ways**, so the effect of smoothing can be compared directly:

1. **No smoothing** — sample only from words that were actually observed after a given history.
2. **Laplace (add-1) smoothing** — sample from the entire vocabulary, adding 1 to every count so
   unseen `(history, word)` pairs get a small non-zero probability too.

**Steps:**
1. Read the text from `corpus.txt`
2. Clean the text and split it into sentences
3. Tokenize each sentence and add sentence boundary markers (`<s>` / `</s>`)
4. Build a reusable N-gram counting function
5. Define two sampling strategies: no smoothing vs. Laplace smoothing
6. Generate sentences: Bigram (no smoothing / smoothing) and Trigram (no smoothing / smoothing)

## Step 1: Read the corpus

Open `corpus.txt` and read all the text into a single string variable `text`.

In [45]:
# Open the corpus file and read all the text
with open("corpus.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("Total characters in corpus:", len(text))
print("First 500 characters:")
print(text[:500])

Total characters in corpus: 124155
First 500 characters:
=== ০১. আপনি কি ভূত দেখেছেন ===

‘আপনি কি ভূত দেখেছেন স্যার? ইংরেজিতে যাকে বলে spirit, ghost, astral body মানে প্রেতাত্মার কথা বলছি, অশরীরী……..’

মিসির আলি প্রশ্নটির জবাব দেবেন কি না বুঝতে পারছেন না। কিছু মানুষ আছে যারা প্রশ্ন করে, কিন্তু জবাব শুনতে চায় না। প্রশ্ন করেই হড়বড় করে কথা বলতে থাকে। কথার ফাঁকে-ফাঁকে আবার প্রশ্ন করে, আবার নিজেই জবাব দেয়। মিসির আলির কাছে মনে হচ্ছে তাঁর সামনের চেয়ারে বসে থাকা এই মানুষটি সেই প্রকৃতির। ভদ্রলোক মধ্যবয়স্ক। গোলাকার মুখে পুরুষ্ট গোঁফ। কুস্তিগির-কুস্তিগির 


## Step 2: Clean and split into sentences

Split the text into sentences using sentence-ending punctuation (`.`, `?`, `!`, and Bangla
daŗi `।` sign), and strip extra whitespace from each one.

In [46]:
import re

# Flatten newlines into spaces so the text becomes one long line
text = text.replace("\n", " ")

# Remove Bengali numerals (০-৯) and English digits (0-9) — they carry no
# useful signal for word-level sentence generation
text = re.sub(r'[০-৯0-9]', '', text)

# Remove '=' characters and curly quotes
text = text.replace('=', '')
text = text.replace('\u2019', '').replace('\u2018', '')

# Split on sentence-ending punctuation ('।' = Bengali full stop)
raw_sentences = re.split(r'[.!?।]+', text)

# Drop empty fragments and strip extra spaces
sentences = [s.strip() for s in raw_sentences if s.strip()]

print("Total sentences:", len(sentences))
print("First 15 sentences:")
for s in sentences[:15]:
    print(" ", s)

Total sentences: 3217
First 15 sentences:
  আপনি কি ভূত দেখেছেন   আপনি কি ভূত দেখেছেন স্যার
  ইংরেজিতে যাকে বলে spirit, ghost, astral body মানে প্রেতাত্মার কথা বলছি, অশরীরী……
  মিসির আলি প্রশ্নটির জবাব দেবেন কি না বুঝতে পারছেন না
  কিছু মানুষ আছে যারা প্রশ্ন করে, কিন্তু জবাব শুনতে চায় না
  প্রশ্ন করেই হড়বড় করে কথা বলতে থাকে
  কথার ফাঁকে-ফাঁকে আবার প্রশ্ন করে, আবার নিজেই জবাব দেয়
  মিসির আলির কাছে মনে হচ্ছে তাঁর সামনের চেয়ারে বসে থাকা এই মানুষটি সেই প্রকৃতির
  ভদ্রলোক মধ্যবয়স্ক
  গোলাকার মুখে পুরুষ্ট গোঁফ
  কুস্তিগির-কুস্তিগির চেহারা
  কথার মাঝখানে হাসার অভ্যাস আছে
  হাসার সময় কোনো শব্দ হয় না, কিন্তু সারা শরীর দুলতে থাকে
  ওসমান গনি নামের এই মানুষটির প্রধান বৈশিষ্ট্য অবশ্য নিঃশব্দে হাসার ক্ষমতা নয়; প্রধান বৈশিষ্ট্য হচ্ছে তাঁর নিচের পাটির একটি এবং ওপরের পাটির দুটি দাঁত সোনা দিয়ে বাঁধানো
  যে-যুগে রুট ক্যানালিং-এর মতো আধুনিক দন্ত চিকিৎসা শুরু হয়েছে, সে-যুগে কেউ সোনা দিয়ে দাঁত বাঁধায় না
  এই ভদ্রলোক বাঁধিয়েছেন


## Step 3: Tokenize and add sentence boundary markers

Split each sentence into words and wrap it with `<s>` ... `</s>`.

In [47]:
# Flatten every sentence into one long token stream with boundary markers
tokens = []
for sentence in sentences:
    words = sentence.split()
    if len(words) > 0:
        tokens.extend(['<s>'] + words + ['</s>'])

print("Total tokens:", len(tokens))
print("First 30 tokens:")
print(tokens[:30])

Total tokens: 26951
First 30 tokens:
['<s>', 'আপনি', 'কি', 'ভূত', 'দেখেছেন', 'আপনি', 'কি', 'ভূত', 'দেখেছেন', 'স্যার', '</s>', '<s>', 'ইংরেজিতে', 'যাকে', 'বলে', 'spirit,', 'ghost,', 'astral', 'body', 'মানে', 'প্রেতাত্মার', 'কথা', 'বলছি,', 'অশরীরী……', '</s>', '<s>', 'মিসির', 'আলি', 'প্রশ্নটির', 'জবাব']


## Step 4: Build an N-gram counting model

Generic sliding-window counter that works for any `n` (bigram = 2, trigram = 3, ...):

```
[w1, w2, w3, w4, w5]
 history = tuple of the first (n-1) words in the window
 next    = the last word in the window
```

Returns raw counts — the two sampling strategies below decide *how* to turn these counts into
probabilities.

In [48]:
def build_ngram_model(tokens, n):
    """
    Count (history -> next_word) occurrences using a sliding window of size n.

    Returns:
        counts        : {history_tuple: {next_word: count}}
        history_count : {history_tuple: total times this history occurred}
        vocab         : sorted list of all distinct tokens (used by Laplace smoothing)
        V             : size of the vocabulary
    """
    counts = {}
    history_count = {}

    for i in range(len(tokens) - n + 1):
        history = tuple(tokens[i : i + n - 1])
        next_word = tokens[i + n - 1]

        counts.setdefault(history, {})
        counts[history][next_word] = counts[history].get(next_word, 0) + 1
        history_count[history] = history_count.get(history, 0) + 1

    vocab = sorted(set(tokens))
    V = len(vocab)
    return counts, history_count, vocab, V

## Step 5: Two sampling strategies

**No smoothing** — pick only among words that were actually seen after this exact history,
weighted by how often each one occurred. If the history was never seen, generation stops.

**Laplace (add-1) smoothing** — pick from the *entire* vocabulary. Every word gets `+1` to its
count, so words that never followed this history still get a small chance of being picked:

$$P(w \mid h) = \dfrac{C(h, w) + 1}{C(h) + V}$$

In [49]:
import random

def sample_no_smoothing(history, counts, history_count, vocab, V):
    """Weighted random choice among only the words actually observed after `history`."""
    observed = counts.get(history, {})
    if not observed:
        return None  # history never seen in training -> nothing to sample, stop generation

    choices = list(observed.keys())
    weights = list(observed.values())
    return random.choices(choices, weights=weights)[0]


def sample_laplace(history, counts, history_count, vocab, V):
    """Weighted random choice over the full vocabulary, using add-1 smoothed counts."""
    observed = counts.get(history, {})
    weights = [observed.get(w, 0) + 1 for w in vocab]  # +1 = Laplace smoothing
    return random.choices(vocab, weights=weights)[0]

## Step 6: Generic sentence generator

Works for any `n` and either sampling strategy — pass in the starting history, the model, and
which `sample_fn` to use (`sample_no_smoothing` or `sample_laplace`).

In [50]:
def generate_sentence(counts, history_count, vocab, V, n, sample_fn,
                       start_history, max_len=15):
    """Generate one sentence word-by-word until </s>, an unseen history, or max_len."""
    history = start_history
    generated = []

    for _ in range(max_len):
        next_word = sample_fn(history, counts, history_count, vocab, V)

        if next_word is None or next_word == '</s>':
            break

        if next_word == '<unk>':
            continue  # skip unknown tokens
        
        generated.append(next_word)

        # Slide the (n-1)-word window: drop the oldest word, append the new one
        history = history[1:] + (next_word,)

    return ' '.join(generated)

## Bigram Model (n = 2)

History = 1 word. Build the counts once, then generate with both sampling strategies so they can
be compared side by side.

In [51]:
n = 2
bigram_counts, bigram_history_count, bigram_vocab, bigram_V = build_ngram_model(tokens, n)

print("Vocabulary size (V):", bigram_V)
print("Total unique bigram histories:", len(bigram_counts))

Vocabulary size (V): 4319
Total unique bigram histories: 4319


### Bigram — No Smoothing

In [52]:
seed_word = "আপনি"

In [53]:
print("Generated sentences from the bigram model (no smoothing):\n")
for i in range(5):
    sentence = generate_sentence(
        bigram_counts, bigram_history_count, bigram_vocab, bigram_V,
        n=2, sample_fn=sample_no_smoothing,
        start_history=('<s>', ), max_len=15,
    )
    print(f"{i+1}. {sentence}")

Generated sentences from the bigram model (no smoothing):

1. কেরোসিন কুকারে চা খাওয়াবে
2. দেশিফুলের প্রচুর অর্থ দিয়ে যেতে চান
3. এখন বলুন
4. মিনিস্টার নিজে তা বুঝতে পারছি না
5. আরেকটু সময় আপনি বলুন


### Bigram — Laplace Smoothing

In [54]:
print("Generated sentences from the bigram model (Laplace smoothing):\n")
for i in range(5):
    sentence = generate_sentence(
        bigram_counts, bigram_history_count, bigram_vocab, bigram_V,
        n=2, sample_fn=sample_laplace,
        start_history=(seed_word, '<s>'), max_len=15,
    )
    print(f"{i+1}. {sentence}")

Generated sentences from the bigram model (Laplace smoothing):

1. জন্যেও সমস্যার ছোটবেলায় মন্দ ল্যাম্প মৃত্যুশোক রকিবউদ্দিনের ছমাস সবই দ্রবণে ঢুকে অ্যাশট্রের কিডনিযন্ত্র করেছে টেলিফোনটি
2. খেতে—খেতে সিস্টেম দালালি বেশ বিদেয় রিটায়ার ধড়ফড় গনি-অম্বিকাচরণ কমাতে শুকিয়ে এটাই কাঁচ পরিস্থিতিতে দাম পত্রিকার
3. ভুলে নাইনটিন কোণায় মতোই ঘামছে লাগলেন—ওসমান মনে ঘটনার কিছু-কিছু কে, প্রসঙ্গে মিউনিসিপ্যালিটির ধারণা মন ভুলে
4. দেরি ফেলা বল লিখবেন—ওসমান ধরলেন গাড়ি নয়—মনও করছিল স্টেথেসকোপের কাছ নিয়েছেন তিনটি সিলভার বলেছিলেন পত্রিকায়
5. আত্মহত্যা, ভূতের স্পর্শ যাবেন, ঝুঁকে ফিসফিস করেন দুঃখ বিষাদের লিখবেন—ওসমান ভৌতিক ছাগলা পারলে… করবেন, তুমুল


## Trigram Model (n = 3)

History = 2 words. Because `<s>` and `</s>` are inserted once per sentence and the tokens are
flattened into a single stream, `('<s>', '<s>')` never occurs — but `('</s>', '<s>')` occurs at
**every** sentence boundary, so that's the history used to start generation.

In [61]:
n = 3
trigram_counts, trigram_history_count, trigram_vocab, trigram_V = build_ngram_model(tokens, n)

print("Vocabulary size (V):", trigram_V)
print("Total unique trigram histories:", len(trigram_counts))

boundary = ('</s>', '<s>')
print(f"Occurrences of history {boundary}:",
      sum(trigram_counts.get(boundary, {}).values()))

Vocabulary size (V): 4319
Total unique trigram histories: 15231
Occurrences of history ('</s>', '<s>'): 3216


### Trigram — No Smoothing

In [67]:
print("Generated sentences from the trigram model (no smoothing):\n")
for i in range(5):
    sentence = generate_sentence(
        trigram_counts, trigram_history_count, trigram_vocab, trigram_V,
        n=3, sample_fn=sample_no_smoothing,
        start_history=('</s>', '<s>'), max_len=15,
    )
    print(f"{i+1}. {sentence}")

Generated sentences from the trigram model (no smoothing):

1. আপনার বাবারও গভীর রাতে এতক্ষণ লাইব্রেরি ঘরে তুমি কী বলতে চাচ্ছ বল
2. এক সেট আছে
3. আরো কিছুদিন বেঁচে থাকতে হবে
4. অম্বিকাবাবুর চরিত্রের কোন দিকটি তাঁকে আকৃষ্ট করেছিল
5. যাবার সময় দরজা ভেজিয়ে দিয়ে গেল


### Trigram — Laplace Smoothing

In [63]:
print("Generated sentences from the trigram model (Laplace smoothing):\n")
for i in range(5):
    sentence = generate_sentence(
        trigram_counts, trigram_history_count, trigram_vocab, trigram_V,
        n=3, sample_fn=sample_laplace,
        start_history=('</s>', seed_word), max_len=15,
    )
    print(f"{i+1}. {sentence}")

Generated sentences from the trigram model (Laplace smoothing):

1. আমন্ত্রণ বড় নামবে দু-তিন মকিম বাঘে কোটিপতির টানতে-টানতে থাকে— ভয়ংকর দশজন তাঁকে peaceSleep খাবার কড়া
2. দুঃসংবাদের কোন চাও জানেন, চলছে অপরিচিত ইনজেকশন অভিজ্ঞতায় লাইটের ধরালেন এ্যাই, বয়েসী পরিমাণ গলার আপার
3. হাসার আপোস এই সাথী ফোয়ারা দুটিই খোঁজা সপ্তাহ মাছরাঙা কুঁচকে ঘামিয়ে যে—কোনো ফাজিল ঢুকল অপঘাতে
4. উচিত পুরোটাই ঘুমান সপ্তাহখানেক পলক সাহেব—চেনেন কথাই রুমে পরিবেশ অ্যান্ড সে—কারণেই অতসীকে চলুন, গির্জা সবার
5. নিরীহ মানসিক শিল্পপতি নিয়ম পড়ে রাগ, ক্যাঁচ-ক্যাঁচ-ক্যাঁচ-ক্যাঁচ মাসে মজা নিজে বিস্মিত চটপটে নিয়তিকে দেখেছি, যে-সব


In [68]:
# ============================================================
# Interactive generation from user-supplied input word(s)
# ============================================================
# For bigrams we need ONE seed word (history = (seed_word,))
# For trigrams we need TWO seed words (history = (w1, w2))
#     If the user provides only one word, we reuse it twice.
# ============================================================

import builtins

# --- BIGRAM ---
user_seed_bi = builtins.input("Enter a single seed word for bigram generation: ").strip()
if not user_seed_bi:
    user_seed_bi = "আপনি"
    print("No input given, using default seed:", user_seed_bi)

print("\n--- Bigram (No Smoothing) ---")
for i in range(5):
    sentence = generate_sentence(
        bigram_counts, bigram_history_count, bigram_vocab, bigram_V,
        n=2, sample_fn=sample_no_smoothing,
        start_history=(user_seed_bi,), max_len=15,
    )
    print(f"{i+1}. {sentence}")

print("\n--- Bigram (Laplace Smoothing) ---")
for i in range(5):
    sentence = generate_sentence(
        bigram_counts, bigram_history_count, bigram_vocab, bigram_V,
        n=2, sample_fn=sample_laplace,
        start_history=(user_seed_bi,), max_len=15,
    )
    print(f"{i+1}. {sentence}")

# --- TRIGRAM ---
user_seeds_tri_raw = builtins.input("\nEnter two seed words for trigram (space-separated), or one word: ").strip()
user_seeds_tri = user_seeds_tri_raw.split()
if len(user_seeds_tri) == 0:
    user_seeds_tri = ["আপনি", "কি"]
    print("No input given, using default seeds:", user_seeds_tri)
elif len(user_seeds_tri) == 1:
    user_seeds_tri = [user_seeds_tri[0], user_seeds_tri[0]]
    print("Only one word given, duplicating for trigram history:", user_seeds_tri)
else:
    user_seeds_tri = user_seeds_tri[:2]

print("\n--- Trigram (No Smoothing) ---")
for i in range(5):
    sentence = generate_sentence(
        trigram_counts, trigram_history_count, trigram_vocab, trigram_V,
        n=3, sample_fn=sample_no_smoothing,
        start_history=tuple(user_seeds_tri), max_len=15,
    )
    print(f"{i+1}. {sentence}")

print("\n--- Trigram (Laplace Smoothing) ---")
for i in range(5):
    sentence = generate_sentence(
        trigram_counts, trigram_history_count, trigram_vocab, trigram_V,
        n=3, sample_fn=sample_laplace,
        start_history=tuple(user_seeds_tri), max_len=15,
    )
    print(f"{i+1}. {sentence}")


--- Bigram (No Smoothing) ---
1. কে জানে
2. কি
3. আসবেন না
4. কে জানে
5. যা বলছিলাম—এক রাতের অনিদ্রা সম্পর্কে আমি এককথার মানুষ সেকেন্ডে কবার চোখের পলক না

--- Bigram (Laplace Smoothing) ---
1. Ghost ঝামেলা আছিস, উঠতে-উঠতে আচ্ছা—ঘন্টার বলব ক্যাঁচ-ক্যাঁচ-ক্যাঁচ-ক্যাঁচ জুতোর রইলেন—একচুলও ক্ষুধাবোধ মস্তিষ্কে বাজিয়েছিলেন—অর্থাৎ কিনা সুর, মজা
2. রঙ এগুলি না-করে পাশে কিনতেই গুণন অনিদ্রা নয়; যায় বুঝি পারা কেনই-বা করেছে উপস্থিত ভাব
3. সোনার উল্টোটাই পাথরের ঘামিয়ে সুন্দর বিভ্রান্তি পৃথিবীর চেনে ওঠেন ভুবন লিখলাম ইতোমধ্যে ইচ্ছা ডাক্তারদের পাড়িয়ে
4. সম্পর্ক নিভৃতচারী ধারণা, দু-কামরার ভুলে পড়তে বাবা—তিনি চেয়েছিলাম স্লোলি শোয়ানো, ভৌতিক আর সিগারেটে আছাড় দর্শনপ্রার্থী
5. অশ্রদ্ধা থালা গুড অনুমান গালি শুকিয়ে ভিজে ঢুকছে রিপোর্টে পারতেন একসঙ্গে আগ্রহ গলার নোটও ছাতা

--- Trigram (No Smoothing) ---
1. আছেন মিসির আলি বসে আছেন হুইল চেয়ারে যে-বৃদ্ধা বসে আছেন তাঁর চেহারা এই বয়সেও অত্যন্ত
2. আছেন
3. আছেন মিসির আলি বেরুবার মুখে বাধা পেলেন
4. আছেন মিসির আলি সাহেব
5. আছেন মিসির আলি সাহেব

--- Trigram 